# CS 459 — Setup Check

**Submit the exported HTML of this notebook to Canvas.**

This is not a coding assignment. It exists to prove your environment works and
to make you look at two or three things you'd otherwise skip.

**Before you start**, the container must be running. In a terminal, from this
folder:

```bash
docker compose up --build
```

Leave that terminal open — the service runs as long as that command does.

If you see an error like:
```
unable to get image 'setup-hello': Cannot connect to the Docker daemon at unix:///Users/{USER}/.docker/run/docker.sock. Is the docker daemon running?
```

Make sure you actually have docker "running".

Then run every cell here, in order.

## 0. Identify yourself

In [ ]:
STUDENT_NAME = "A"      # e.g. "Ada Lovelace"
STUDENT_EMAIL = "A"     # your @u.boisestate.edu address

# If you remapped the port in compose.yaml, change this to match.
SERVICE_PORT = 8000

assert STUDENT_NAME and STUDENT_EMAIL, "Fill in your name and email, then re-run this cell."
print(f"{STUDENT_NAME} <{STUDENT_EMAIL}>")

## 1. Which Python is this?

The most common setup failure in this course is a notebook running against a
*different* Python than the one your packages went into. Jupyter doesn't always
pick the interpreter you expect.

`sys.executable` is ground truth. It should point inside this folder's `.venv`.
If it points at `/usr/bin/python3`, `C:\Python312\`, or an Anaconda folder,
your kernel is wrong — see Troubleshooting in `instructions.md`.

In [ ]:
import platform
import sys

in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)

print(f"Python version : {platform.python_version()}")
print(f"Interpreter    : {sys.executable}")
print(f"Platform       : {platform.platform()}")
print(f"In a venv?     : {in_venv}")

if not in_venv:
    print("\n>>> PROBLEM: this kernel is NOT in a virtual environment.")
    print(">>> Your packages are probably somewhere this kernel can't see.")
    print(">>> Fix: instructions.md -> 'Jupyter is using the wrong Python'.")

## 2. Is git configured?

In [ ]:
import shutil
import subprocess


def git(*args):
    if not shutil.which("git"):
        return ""
    return subprocess.run(["git", *args], capture_output=True, text=True).stdout.strip()


git_version = git("--version")
git_name = git("config", "--get", "user.name")
git_email = git("config", "--get", "user.email")

print(f"git       : {git_version or 'NOT FOUND ON PATH'}")
print(f"user.name : {git_name or '(not set)'}")
print(f"user.email: {git_email or '(not set)'}")

## 3. Talk to the container

Everything above ran on your laptop. This cell reaches across a network boundary
into a process running inside a container.

`localhost:8000` on your machine forwards to port 8000 *inside* the container,
because of the `ports:` line in `compose.yaml`. Without that mapping the service
would be running perfectly and be completely unreachable.

In [ ]:
import requests

BASE_URL = f"http://localhost:{SERVICE_PORT}"

try:
    response = requests.get(BASE_URL, timeout=5)
    response.raise_for_status()
    container = response.json()
except requests.exceptions.ConnectionError:
    print(f">>> PROBLEM: nothing is listening on {BASE_URL}")
    print(">>>")
    print(">>> Checklist, in order:")
    print(">>>   1. Is your container runtime running? (Docker Desktop whale icon,")
    print(">>>      or `colima status`, or however you started it)")
    print(">>>   2. In this folder, did you run:  docker compose up --build")
    print(">>>   3. Did that terminal print 'Serving on http://0.0.0.0:8000'?")
    print(">>>   4. Run `docker compose ps` -- does it show the container running?")
    print(">>>   5. If you remapped the port, update SERVICE_PORT above.")
    raise SystemExit("Container unreachable. Fix the above, then re-run this cell.")

for key, value in container.items():
    print(f"  {key:<15}: {value}")

### What you're looking at

Compare two things:

- `hostname` above is the **container's** ID — not your laptop's name. Run the
  next cell to see them side by side.
- `python_version` above is the container's Python, from the `FROM python:3.12-slim`
  line in the `Dockerfile`. It has nothing to do with the Python running this
  notebook, and the two can differ freely without interfering.

In [ ]:
import socket

print(f"Your laptop's hostname : {socket.gethostname()}")
print(f"Container's hostname   : {container['hostname']}")
print(f"Container reports it's in Docker: {container['in_container']}")
print()
print(f"Your Python     : {platform.python_version()}")
print(f"Container Python: {container['python_version']}")
print()

if container["in_container"]:
    print("Two Pythons, same machine, neither one aware of the other.")
    print("That isolation is the entire reason containers exist.")
else:
    print(">>> PROBLEM: whatever answered you is NOT running in a container.")
    print(">>> You most likely started app.py directly instead of using Docker.")
    print(">>> Stop it, then run:  docker compose up --build")

## 4. One short question

**Question:** The `Dockerfile` runs the server with `--host 0.0.0.0` rather than
`127.0.0.1`, and there's a comment explaining why. In your own words: what would
happen if it bound to `127.0.0.1` instead, and why would that be confusing to
debug?

*3–4 sentences. Graded on reasoning, not on being right.*

In [ ]:
ANSWER = """
(your answer here)
"""

print(ANSWER.strip())
assert len(ANSWER.strip()) > 80, "Give this a real answer -- a few sentences."

## 5. Submission receipt

Run this last, then **File → Save**, then
**File → Export Notebook As → HTML**. Upload that HTML to Canvas.

Export *after* running everything, or your outputs won't be in the file.

In [ ]:
import json
from datetime import datetime, timezone

receipt = {
    "student": STUDENT_NAME,
    "email": STUDENT_EMAIL,
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "local": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
        "in_venv": in_venv,
    },
    "git": {"user_name": git_name, "user_email": git_email},
    "container": container,
    "answered": len(ANSWER.strip()) > 80,
}

print("=" * 64)
print("CS 459 -- SETUP CHECK RECEIPT")
print("=" * 64)
print(json.dumps(receipt, indent=2))
print("=" * 64)

ok = (
    in_venv
    and bool(git_name and git_email)
    and container.get("in_container") is True
    and receipt["answered"]
)
print("COMPLETE -- export to HTML and submit." if ok
      else "INCOMPLETE -- scroll up, something above did not pass.")